In [ ]:
import numpy as np

from ase import io, units, Atoms
from ase.geometry import get_distances
from ase.neighborlist import PrimitiveNeighborList
from ase.data import covalent_radii
from ase.constraints import FixSubsetCom
from ase.md.nose_hoover_chain import NoseHooverChainNVT

import pfp_api_client
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

estimator = Estimator(calc_mode=EstimatorCalcMode.PBE_PLUS_D3, model_version='v9.0.0')
calculator = ASECalculator(estimator)

print(f'pfp_api_client: {pfp_api_client.__version__}')

In [ ]:
directory   = '/home/jovyan/Matlantis/VN_111/363K'  
temperature = 363                
timestep    = 2.5                  
tdamp       = 250.0                
md_steps    = 2500               
log_steps   = 10

In [ ]:
def remove_desorbed(atoms):
    extreme_mask = atoms.get_positions()[:, 2] > 60.0
    if extreme_mask.any():
        print(f"Removing {extreme_mask.sum()} stranded atoms above 60 Å")
        atoms = atoms[~extreme_mask]
    radii = np.array([covalent_radii[z] for z in atoms.numbers])
    nl = PrimitiveNeighborList(radii * 1.5, skin=0.0, self_interaction=False, bothways=True)
    nl.update(atoms.pbc, atoms.get_cell(complete=True), atoms.get_positions())

    mask = (atoms.get_positions()[:, 2] > 24.0)
    keep_mask = np.ones_like(mask)

    molecule_idxs = -np.ones_like(atoms.get_tags())

    for idx in np.nonzero(mask)[0]:
        neighbours = set((idx,))

        while neighbours:
            nidx = neighbours.pop()

            if molecule_idxs[nidx] == -1:
                molecule_idxs[nidx] = idx
                new_neighbours = set(nl.get_neighbors(nidx)[0])
                neighbours.update(new_neighbours)

        if np.any(molecule_idxs == idx):
            molecule = atoms[molecule_idxs == idx]
            if molecule.get_momenta()[:, 2].mean() > 0.0 and molecule.get_positions()[:, 2].min() > 17.5:
                print(f'Deleting molecule {idx} {molecule.get_chemical_formula()} with upwards momenta {molecule.get_momenta()[:, 2].mean()}')
                keep_mask[molecule_idxs == idx] = False

    atoms = atoms[keep_mask]
    return atoms

In [ ]:
def add_adsorbate(atoms, molecule, rng, tag=0):
    while True:
        newatoms = molecule.copy()
        x, y = rng.random(2)
        rot_vec = rng.normal(size=3)
        rot_ang = rng.uniform(0, 360.0)

        trans = atoms.cell.cartesian_positions([x, y, 0.0]) + [0.0, 0.0, 26.0]
        
        newatoms.rotate(rot_ang, rot_vec)
        newatoms.translate(trans)
        
        phi = rng.uniform(-1.0, 1.0, size=len(newatoms)) * np.pi/12
        th = rng.uniform(0.0, 2*np.pi, size=len(newatoms))
        
        initial_vel = 750.0 * np.array([np.cos(th)*np.sin(phi), np.sin(th)*np.sin(phi), -np.cos(phi)]).T * (units.m / units.s)
        
        newatoms.set_velocities(initial_vel)
        newatoms.set_tags(tag)
        
        _disps, dists = get_distances(atoms.positions, newatoms.positions, atoms.cell, atoms.pbc)

        if dists.min() > 1.5:
            break

    print(f'Added species {newatoms.get_chemical_formula()} at {trans[0]:6.2f} {trans[1]:6.2f} {trans[2]:6.2f}')
    atoms += newatoms
    return atoms

In [ ]:
rng = np.random.default_rng()
atoms = io.read(f'{directory}/surface_relaxed.traj', index=-1)
atoms.set_pbc([True, True, False])

log_file = f'{directory}/MD.log'

etch_species = io.read(f'{directory}/reactant.traj', index=-1)

for i in range(0, 16):
    output_file = f'{directory}/{i:0>2d}_MD.traj'
    trajectory_file = f'{directory}/{i:0>2d}_MD-trajectory.traj'
    
    
    atoms = remove_desorbed(atoms)
    n_before = len(atoms)
    atoms = add_adsorbate(atoms, etch_species, rng, tag=(i + 1))

    atoms.calc = calculator
    atoms.set_constraint(FixSubsetCom(np.nonzero(atoms.get_tags() == 0)[0]))

    dyn = NoseHooverChainNVT(atoms,
                             timestep=timestep * units.fs, temperature_K=temperature, tdamp=tdamp * units.fs,
                             logfile=log_file, trajectory=trajectory_file, loginterval=log_steps)

    status = dyn.run(md_steps)
    atoms = remove_desorbed(atoms)
    n_after = len(atoms)
    
    # If the added O was immediately removed, surface is saturated - stop
    if n_after <= n_before:
        print(f"Surface saturated at round {i} - stopping")
        io.write(output_file, atoms)
        break
    io.write(output_file, atoms)

In [ ]:
# Run longer timescales, larger supercell, higher temperature
# Increase vacuum region to solve periodicity problem